# Part 8: Neural Network Foundations

**Course:** 2026 KMITL Data Analytics

A neural network combines simple calculations to learn a useful mapping from inputs to outputs. We will calculate neurons by hand, build a small network with NumPy, and train it using gradients. No deep-learning framework or dataset download is needed.


## Learning objectives

1. Place deep learning inside machine learning and distinguish shallow and deep networks.
2. Calculate a weighted sum and explain inputs, weights and bias.
3. Compare ReLU, sigmoid and softmax activations.
4. Trace an input through hidden and output layers.
5. Connect loss, backpropagation and an optimizer.
6. Explain batches, iterations and epochs, then read a training-loss chart.


## Required imports

NumPy provides arrays and matrix multiplication. Matplotlib draws activation curves and the flow of information. A fixed random generator makes the small training example repeatable.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
print('NumPy neural-network lesson ready.')


## Dataset introduction

We start with one input vector [2, 3]. Later we use the four rows of **exclusive OR (XOR)**: [0, 0] and [1, 1] have target 0; [0, 1] and [1, 0] have target 1. The target is 1 when exactly one input is 1. A straight decision boundary cannot separate these four points correctly. This makes XOR a small illustration of why a nonlinear hidden layer is useful.

We train and inspect the same four cases to understand optimization, not to measure performance on new data. A real task needs separate training, validation and test data.

**Machine learning** includes methods such as trees, linear models and neural networks. **Deep learning** uses neural networks with multiple learned layers to build representations. A shallow network usually has one hidden layer; a deep network has several. Depth is not automatically better: the data, resources and task determine what is useful.


## 1. One artificial neuron

An artificial neuron is a mathematical unit, not a realistic brain cell. It multiplies each input by a **weight**, adds the results and a **bias**, then applies an activation function. Weights control input influence; bias shifts the result even when inputs are zero.

For inputs [2, 3], weights [0.5, −1] and bias 1, the weighted sum is 2 × 0.5 + 3 × (−1) + 1 = **−1**. The sigmoid activation, explained next, turns this into about **0.269**.

For n inputs, x_j is input j, w_j is its weight, b is bias, z is the weighted sum, and a is the activation function:

`z = sum(x_j * w_j for j = 1, ..., n) + b`

`output = a(z)`


In [ ]:
def sigmoid(values):
    # logaddexp avoids overflow for very large positive or negative inputs.
    return np.exp(-np.logaddexp(0, -np.asarray(values, dtype=float)))

inputs = np.array([2., 3.])
weights = np.array([0.5, -1.])
bias = 1.
weighted_sum = inputs @ weights + bias
neuron_output = sigmoid(weighted_sum)
print('Weighted sum:', weighted_sum, '| sigmoid output:', round(float(neuron_output), 3))
assert np.isclose(weighted_sum, -1)
assert np.isclose(neuron_output, 0.2689414214)


## 2. Activation functions

Without nonlinear activations, stacking weighted-sum layers still gives a single linear mapping. Activations let the network represent more complex patterns.

- **ReLU** keeps positive values and replaces negative values with zero. Inputs [−2, 0, 3] become [0, 0, 3]. In general, `ReLU(z) = max(0, z)`. It is common in hidden layers; a unit that always stays negative can stop receiving useful gradients.
- **Sigmoid** maps one score to the interval (0, 1). At z = 0, it gives 1/(1 + 1) = **0.5**; at z = −1 it gives about **0.269**. In general, `sigmoid(z) = 1 / (1 + exp(-z))`, where exp is the exponential function. It can represent one binary-class probability, but does not guarantee calibrated probabilities. Large-magnitude inputs give very small gradients.
- **Softmax** transforms a vector of class scores into nonnegative probabilities that sum to one. Scores [0, ln(2), ln(3)] have exponentials [1, 2, 3], giving **[1/6, 2/6, 3/6]**. Here ln is the natural logarithm. For score z_i among c classes, `softmax(z)_i = exp(z_i) / sum(exp(z_j) for j = 1, ..., c)`. Subtracting the largest score first keeps the answer unchanged and avoids overflow.

Softmax acts on a group of scores, unlike an element-wise activation. We plot one three-class output vector rather than pretending it is a single-input curve.


In [ ]:
def softmax(scores):
    shifted = np.asarray(scores) - np.max(scores, axis=-1, keepdims=True)
    exponentials = np.exp(shifted)
    return exponentials / exponentials.sum(axis=-1, keepdims=True)

scores = np.log([1., 2., 3.])
probabilities = softmax(scores)
assert np.allclose(probabilities, [1 / 6, 2 / 6, 3 / 6])
assert np.allclose(softmax(scores + 1000), probabilities)
values = np.linspace(-6, 6, 200)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
axes[0].plot(values, np.maximum(0, values), label='ReLU')
axes[0].set(title='ReLU', xlabel='Weighted sum z', ylabel='Activation')
axes[1].plot(values, sigmoid(values), label='Sigmoid', color='darkorange')
axes[1].set(title='Sigmoid', xlabel='Weighted sum z', ylabel='Activation')
axes[2].bar(['Class 0', 'Class 1', 'Class 2'], probabilities, label='Softmax')
axes[2].set(title='Softmax of [0, ln(2), ln(3)]', xlabel='Class', ylabel='Probability', ylim=(0, 1))
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.show()


**Read the charts:** ReLU is flat for negative inputs and rises for positive ones. Sigmoid changes most around zero and becomes flat near its extremes. The softmax bars sum to one; increasing one class score affects the other classes' probabilities too. Use sigmoid for a binary output and softmax for mutually exclusive multiple classes. Independent labels require a different output setup.


## 3. Layers and a forward pass

The **input layer** receives features. A **hidden layer** computes an intermediate representation. The **output layer** produces the prediction. A **forward pass** sends values through these layers in order.

For input [1, 0], let two hidden neurons have weights [1, 1] and [−1, 1], both with zero bias. Their sums are [1, −1], so ReLU gives **[1, 0]**. Output weights [2, −1] and bias −0.5 give 1 × 2 + 0 × (−1) − 0.5 = **1.5**, and sigmoid gives about **0.818**.

For a batch matrix X, hidden weights W1 and bias b1, output weights W2 and bias b2, hidden values H and prediction P, the general forward pass is:

`H = activation(X @ W1 + b1)`

`P = output_activation(H @ W2 + b2)`

The `@` operator is matrix multiplication. Each row is one example, and each column of a weight matrix supplies weights for one neuron. Shapes must agree.


In [ ]:
sample_inputs = np.array([[1., 0.]])
hidden_weights = np.array([[1., -1.], [1., 1.]])
hidden_bias = np.zeros(2)
output_weights = np.array([[2.], [-1.]])
output_bias = np.array([-0.5])
hidden_values = np.maximum(0, sample_inputs @ hidden_weights + hidden_bias)
prediction = sigmoid(hidden_values @ output_weights + output_bias)
assert np.allclose(hidden_values, [[1, 0]])
assert np.allclose(prediction, sigmoid(1.5))
print('Shapes:', sample_inputs.shape, '->', hidden_values.shape, '->', prediction.shape)
fig, ax = plt.subplots(figsize=(11, 3))
positions = [0.12, 0.5, 0.88]
labels = ['Input layer: [1, 0]', 'Hidden ReLU: [1, 0]', f'Output sigmoid: {prediction.item():.3f}']
for position, label in zip(positions, labels):
    ax.text(position, 0.5, label, ha='center', va='center', transform=ax.transAxes,
            bbox={'boxstyle': 'round,pad=0.7', 'facecolor': 'lightblue'})
for start, end in [(0.24, 0.37), (0.63, 0.76)]:
    ax.annotate('', xy=(end, 0.5), xytext=(start, 0.5), xycoords='axes fraction',
                arrowprops={'arrowstyle': '->', 'lw': 2})
ax.set_title('Forward pass: learned transformations connect layers')
ax.axis('off')
plt.tight_layout()
plt.show()


**Read the diagram:** information moves left to right through two learned transformations. The input has two features, the hidden layer has two units, and the output has one unit. A value of 0.818 is a model output, not evidence that these hand-chosen weights solve a real task. Training must adjust weights using data.


In [ ]:
first_weights = np.linspace(-2, 2, 100)
changed_outputs = sigmoid(inputs[0] * first_weights + inputs[1] * weights[1] + bias)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(first_weights, changed_outputs, label='Only first weight changes')
ax.scatter([weights[0]], [neuron_output], color='red', label='Original weight 0.5')
ax.set(title='One weight changes one neuron output', xlabel='First weight', ylabel='Sigmoid output')
ax.legend()
plt.tight_layout()
plt.show()


**Read the weight curve:** we reuse inputs [2, 3], keep the second weight at −1 and the bias at 1. Increasing the first weight raises the sum because its input is positive. The output changes slowly in saturated regions of sigmoid. The red point recovers our original output, about 0.269. A negative input would reverse the direction of the effect.


## 4. A loss tells training what to improve

A **loss function** measures how wrong predictions are. For binary targets, binary cross-entropy strongly penalizes confident wrong answers. When the true target is 1, predicting 0.8 gives −ln(0.8) ≈ **0.223**; predicting 0.2 gives −ln(0.2) ≈ **1.609**. Smaller is better.

For target y (0 or 1), predicted positive-class probability p, and natural logarithm ln:

`loss = -(y * ln(p) + (1 - y) * ln(1 - p))`

We average this loss across the batch. The code clips probabilities away from exact zero and one to avoid taking a logarithm of zero. Accuracy uses a threshold and can stay unchanged while loss improves; they measure different things.


In [ ]:
def binary_loss(actual, predicted):
    safe_predictions = np.clip(predicted, 1e-12, 1 - 1e-12)
    return -np.mean(actual * np.log(safe_predictions) + (1 - actual) * np.log(1 - safe_predictions))

print('Correct confident prediction:', binary_loss(np.array([1]), np.array([0.8])))
print('Wrong confident prediction:', binary_loss(np.array([1]), np.array([0.2])))
assert binary_loss(np.array([1]), np.array([0.8])) < binary_loss(np.array([1]), np.array([0.2]))


## 5. Backpropagation and the optimizer

**Backpropagation** applies the chain rule to calculate how each parameter affects loss. It sends derivatives backward through the operations used in the forward pass. It does not itself update the parameters. The **optimizer** uses those derivatives to choose updates.

For one sigmoid output with binary cross-entropy, the derivative with respect to its input score is predicted probability minus target. If p = 0.8 and y = 1, that derivative is **−0.2**. If the hidden value feeding its weight is 0.5, the weight derivative is −0.2 × 0.5 = **−0.1**. With learning rate 0.1 and current weight 1, gradient descent gives 1 − 0.1 × (−0.1) = **1.01**.

For a parameter w, its loss derivative g and learning rate a, the update is `new_w = w - a * g`. A large rate may overshoot; a small rate can learn slowly.

Our training network uses sigmoid in the hidden layer too, so we can explain its derivative simply: an activation h = 0.5 gives 0.5 × (1 − 0.5) = **0.25**. In general, the sigmoid derivative is `h * (1 - h)`. Multiply the incoming derivative by this local derivative to move backward.

For N examples, predicted values P, targets Y, hidden values H and output weights W2, the code uses:
- `output_error = (P - Y) / N` for the mean loss.
- `output_gradient = H.T @ output_error` for output weights.
- `hidden_error = (output_error @ W2.T) * H * (1 - H)` for hidden scores.
- `hidden_gradient = X.T @ hidden_error` for hidden weights; bias gradients sum errors over rows.

`.T` transposes rows and columns. All gradients must be calculated with the same current weights before updating any parameter.


### Batch, iteration and epoch

A **batch** is the set of examples used for one update. An **iteration** is one update step. An **epoch** is one pass through the training examples. With 10 examples and batch size 4, one epoch has batches of 4, 4 and 2: **3 iterations**.

The number of iterations per epoch is `ceil(number_of_examples / batch_size)` when the final partial batch is kept; ceil rounds up. In our XOR example all four rows form one full batch, so one iteration is one epoch. Keras will handle mini-batches in Part 9.


In [ ]:
train_inputs = np.array([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
train_targets = np.array([[0.], [1.], [1.], [0.]])
rng = np.random.default_rng(RANDOM_STATE)
first_weights = rng.normal(0, 1, size=(2, 4))
first_bias = np.zeros((1, 4))
last_weights = rng.normal(0, 1, size=(4, 1))
last_bias = np.zeros((1, 1))
learning_rate = 1.0
loss_history = []

for step in range(3000):
    hidden = sigmoid(train_inputs @ first_weights + first_bias)
    predicted = sigmoid(hidden @ last_weights + last_bias)
    loss_history.append(binary_loss(train_targets, predicted))
    output_error = (predicted - train_targets) / len(train_inputs)
    last_gradient = hidden.T @ output_error
    last_bias_gradient = output_error.sum(axis=0, keepdims=True)
    hidden_error = (output_error @ last_weights.T) * hidden * (1 - hidden)
    first_gradient = train_inputs.T @ hidden_error
    first_bias_gradient = hidden_error.sum(axis=0, keepdims=True)
    last_weights -= learning_rate * last_gradient
    last_bias -= learning_rate * last_bias_gradient
    first_weights -= learning_rate * first_gradient
    first_bias -= learning_rate * first_bias_gradient

hidden = sigmoid(train_inputs @ first_weights + first_bias)
predicted = sigmoid(hidden @ last_weights + last_bias)
loss_history.append(binary_loss(train_targets, predicted))
print('Inputs | target | probability')
print(np.column_stack([train_inputs, train_targets, predicted]).round(3))
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(len(loss_history)), loss_history, label='Full-batch training loss')
ax.set(title='Learning XOR with a 2–4–1 network', xlabel='Completed updates', ylabel='Binary cross-entropy')
ax.legend()
plt.tight_layout()
plt.show()


**Read the loss chart:** the first point is the initial loss before any update. Later points use learned weights. The fixed-seed network learns all four XOR cases and reduces its training loss. A low training loss is not a generalization score: no validation or test set exists in this small demonstration. Different initial weights or learning rates can change the path and convergence speed.


## Common mistakes

- Stacking linear layers without nonlinear activations and expecting nonlinear behavior.
- Forgetting the bias or mixing up the example and feature axes.
- Applying softmax across different examples rather than across classes.
- Calling backpropagation an optimizer: one computes derivatives, the other updates parameters.
- Updating output weights before calculating hidden gradients.
- Confusing an epoch with a batch update, or calling training accuracy test accuracy.


In [ ]:
assert np.isfinite(loss_history).all()
assert loss_history[-1] < loss_history[0] * 0.1
assert np.array_equal((predicted >= 0.5).astype(int), train_targets.astype(int))
assert np.allclose(softmax([[1., 2., 3.], [1000., 1001., 1002.]]).sum(axis=1), 1)
assert np.isfinite(sigmoid([-1000., 0., 1000.])).all()
# Check one backpropagated weight derivative against a small numerical change.
hidden = sigmoid(train_inputs @ first_weights + first_bias)
predicted = sigmoid(hidden @ last_weights + last_bias)
analytic_gradient = (hidden.T @ ((predicted - train_targets) / len(train_inputs)))[0, 0]
step_size = 1e-5
higher_weights = last_weights.copy()
lower_weights = last_weights.copy()
higher_weights[0, 0] += step_size
lower_weights[0, 0] -= step_size
higher_loss = binary_loss(train_targets, sigmoid(hidden @ higher_weights + last_bias))
lower_loss = binary_loss(train_targets, sigmoid(hidden @ lower_weights + last_bias))
numerical_gradient = (higher_loss - lower_loss) / (2 * step_size)
assert np.isclose(analytic_gradient, numerical_gradient, atol=1e-7)
print('Neural-network foundation checks passed.')


## Summary

A neuron combines inputs, weights, bias and activation. Hidden layers let networks build nonlinear representations. A forward pass produces predictions; a loss measures mistakes; backpropagation computes gradients; an optimizer updates parameters. Batches and epochs describe how data is processed. The next lesson uses Keras to manage these operations and evaluate a model on separate data.
